# Finite-sample risk-aware importance sampling for real Wi-Fi indoor localization

**Full experiment — one run, no synthetic substitutions.**  
Dataset: [UJIIndoorLoc, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/310/ujiindoorloc), 19,937 official reference/training fingerprints and 1,111 official held-out fingerprints, 520 access points. The experiment estimates horizontal positions using real measurements, not simulated radio signals. It is a retrospective benchmark pipeline, not a deployed tracking system.

### Run in Colab
1. Open this notebook with the Google account that has GPU access. Select **Runtime → Change runtime type → T4 GPU (or available GPU)**.
2. Select **Runtime → Run all**. The full configuration is fixed below; no pilot-mode switch or manual intervention is necessary unless the UCI download fails. If that happens, the notebook prompts for the official ZIP.
3. Keep the runtime connected. The final cell creates and downloads `UJIIndoorLoc_Finite_Sample_Risk_Results.zip`. It also remains in `/content/`. Intermediate oracle and teacher arrays are saved as cells finish; these are records, not automatic restart checkpoints. No Google Drive mount or other account authorization is needed.

### Question and protocol (fixed before the official test)
For a batch of Wi-Fi fingerprints $x_i$, a probabilistic fingerprint model defines a finite discrete posterior over surveyed locations $z$. A neural categorical proposal $q(z\mid x_i)$ is trained on the model-fitting split; a uniform defensive component guarantees support. Estimate the posterior horizontal mean via SNIS. For observation $i$, the objective is $R_i(K)=\frac12\mathbb E\|\hat\mu_{i,K}-\mu_i\|_2^2$. Use an average of 48 target likelihood evaluations per record in batches of up to 32, with integer bounds 4–120. An exact enumeration of the finite model posterior supplies the reference; it is **not** supplied to any deployable policy. This is a real-data finite-support modeling choice, not a continuous ground-truth posterior.

Separate fitting, calibration, risk-teacher, and policy-validation splits are made from the official training records. User-disjoint development partitions are preferred; if infeasible, the fallback is explicitly recorded. The official held-out data is never used to fit models, select the cheap comparator, select temperature, or choose the shrinkage coefficient. The fitted target uses a uniform prior over observed training-map locations and a tempered Bernoulli/Gaussian fingerprint likelihood; calibration uses development location error.

**Primary contrast:** pooled finite-sample oracle (two independent 256-repetition banks) versus a cheap baseline selected on development records. The evaluation bank has 512 fresh paired repetitions per official test record. Gate: at least 2% relative MSE gain with conditional 95% interval excluding zero. This is a preliminary benchmark-specific gate, not population inference or a guaranteed bound. Learned risk and fixed validation-calibrated shrinkage are secondary; neither is assumed to win. Independent oracle banks quantify allocation instability.

**Practical comparisons:** exact enumeration, direct neural proposal mean, and 5-nearest-neighbor fingerprint regression. Benchmark synchronized online runtime including proposal, allocator and actual sampled target evaluations, not precomputed references. Compare coordinate error against recorded positions separately from posterior estimation error. Coordinates are centered native projected units, not asserted physical meters. The direct baselines also report building/floor accuracy; the primary sampling statistic is horizontal posterior mean, which can fall between modes or buildings. Full proposal probabilities across survey nodes have a cost, even when target evaluations are capped. Exact enumeration may be faster than importance sampling here. A negative result is scientifically informative.

### Outputs
CSV comparisons, validation choices, oracle gate, per-batch and device/user-stratified descriptive results, online timings, learned and exact risk curves, model parameters, an example online batch with coordinates, six figures, and raw paired Monte Carlo metrics. The archive also records the data hash, seeds, protocol, and fitted parameters. Do not report a gain as a completed result until this real-data notebook has actually run.

In [ ]:
# Install only missing dependencies (standard Colab usually has all of these).
import importlib.util, subprocess, sys
for module, package in [('numba','numba'),('sklearn','scikit-learn'),('scipy','scipy'),('joblib','joblib'),('pandas','pandas'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

## Configuration

All parameters below are already set for the full study. GPU accelerates fitting and Monte Carlo banks; bounded integer allocation and the risk regressors run on CPU.

In [ ]:
import os,io,zipfile,urllib.request,json,time,hashlib,math
from pathlib import Path
import numpy as np,pandas as pd,torch,matplotlib.pyplot as plt
from torch import nn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import t
from numba import njit
from IPython.display import display
SEED=20261123; TEST_FIXTURE=False
NOPT=256 if not TEST_FIXTURE else 8; NEVAL=512 if not TEST_FIXTURE else 12; NTEACH=384 if not TEST_FIXTURE else 8
NVAL=256 if not TEST_FIXTURE else 8; EPOCHS=30 if not TEST_FIXTURE else 2
KMIN,KMAX,KAVG=4,120,48; KS=np.arange(KMIN,KMAX+1); BATCH=32 if not TEST_FIXTURE else 8
OUT=Path('/content/wifi_results' if Path('/content').exists() else ('/tmp/wifi_fixture_results' if TEST_FIXTURE else '/tmp/wifi_results')); OUT.mkdir(exist_ok=True)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); torch.manual_seed(SEED); np.random.seed(SEED)
if DEV.type=='cuda': torch.cuda.manual_seed_all(SEED)
else: torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32=False
print('Device:',DEV, 'FULL REAL DATA STUDY')
start_total=time.perf_counter()
def T(x,dtype=torch.float32): return torch.as_tensor(x,dtype=dtype,device=DEV)
def sync():
    if DEV.type=='cuda': torch.cuda.synchronize()

## Data download

Download the original official files directly from UCI. Verify the expected row counts and fingerprint columns. Never replace the real dataset with simulated data.

In [ ]:
URLS=['https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip','https://archive.ics.uci.edu/ml/machine-learning-databases/00310/UJIIndoorLoc.zip']
def readzip(blob):
    d={}
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for n in z.namelist():
            if n.lower().endswith('.zip'): d.update(readzip(z.read(n)))
            elif n.lower().endswith('trainingdata.csv'): d['train']=pd.read_csv(io.BytesIO(z.read(n)))
            elif n.lower().endswith('validationdata.csv'): d['test']=pd.read_csv(io.BytesIO(z.read(n)))
    if not d: raise ValueError('No official CSVs found')
    return d
cache=OUT/'uji.zip'; data=None; errors=[]
for url in ([None] if cache.exists() else [])+URLS:
    try:
        if url is None: blob=cache.read_bytes()
        else:
            with urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'}),timeout=90) as r: blob=r.read()
        data=readzip(blob); assert set(data)=={'train','test'}; cache.write_bytes(blob); break
    except Exception as e: data=None;errors.append(str(e))
if data is None:
    print('Download failed:',errors,'Upload the official ZIP from https://archive.ics.uci.edu/dataset/310/ujiindoorloc')
    from google.colab import files
    blob=next(iter(files.upload().values()));data=readzip(blob);cache.write_bytes(blob)
train,test=data['train'],data['test'];sha=hashlib.sha256(blob).hexdigest();assert (len(train),len(test))==(19937,1111)
for f in (train,test): f.columns=f.columns.str.strip().str.upper()
WAPS=sorted(c for c in train if c.startswith('WAP')); A=len(WAPS); LOC=['BUILDINGID','FLOOR','LONGITUDE','LATITUDE']
assert A==(12 if TEST_FIXTURE else 520); assert not train[WAPS+LOC].isna().any().any() and not test[WAPS+LOC].isna().any().any()
print('Training / official test:',len(train),len(test),'APs:',A,'SHA256:',sha)

## Development split and survey map

The survey map and generative model use fit records only. The other development partitions are used for calibration, risk supervision and policy selection. Official test records are held out from model selection.

In [ ]:
# Select disjoint user groups using original training data only. If infeasible, explicitly disclose row split.
def split(f):
    ids=np.arange(len(f));u=f.USERID.to_numpy();g=np.unique(u);s=(f.BUILDINGID.astype(str)+'_'+f.FLOOR.astype(str)).to_numpy();rng=np.random.default_rng(SEED);best=None;bs=1e9
    for _ in range(750):
        if len(g)<8:break
        q=rng.permutation(g);a=round(.6*len(g));b=max(1,round(.13*len(g))); parts=[ids[np.isin(u,h)] for h in [q[:a],q[a:a+b],q[a+b:a+2*b],q[a+2*b:]]]; sizes=np.array([len(x) for x in parts]);score=np.sum((sizes/len(f)-[.60,.13,.13,.14])**2)
        if min(sizes)>=(20 if TEST_FIXTURE else 250) and set(s[parts[0]])==set(s) and score<bs:best=parts;bs=score
    if best is not None:return best,'user-disjoint development'
    a,b=train_test_split(ids,test_size=.4,random_state=SEED,stratify=s);c,b=train_test_split(b,train_size=1/3,random_state=SEED+1,stratify=s[b]);d,e=train_test_split(b,test_size=.5,random_state=SEED+2,stratify=s[b]);return [a,c,d,e],'stratified rows; development users can overlap'
parts,split_type=split(train); fit,cal,teacher,val=[train.iloc[p].reset_index(drop=True) for p in parts]
def cap(f,n,seed):return f.sample(n=n,random_state=seed).reset_index(drop=True) if len(f)>n else f
cal=cap(cal,1600,SEED+3);teacher=cap(teacher,1000,SEED+4);val=cap(val,512,SEED+5)
assert len(np.unique(np.concatenate(parts)))==len(train)
print(split_type, [len(x) for x in [fit,cal,teacher,val,test]])
key=fit[LOC].to_numpy(np.float64);key[:,2:]=np.round(key[:,2:],2);nodes,labels=np.unique(key,axis=0,return_inverse=True);S=len(nodes);origin=nodes[:,2:].mean(0);XY=T(nodes[:,2:]-origin);groups,gi=np.unique(nodes[:,:2],axis=0,return_inverse=True);G=len(groups);GROUP=T(np.eye(G)[gi])
print('Survey locations:',S,'building-floor groups:',G)
def encode(f):
    raw=f[WAPS].to_numpy(np.float32);d=(raw!=100).astype(np.float32);r=np.where(d>0,(raw+75)/20,0);return d,r
def arrays(f):
    d,r=encode(f);gt=f[['LONGITUDE','LATITUDE']].to_numpy(np.float64)-origin;grp=np.full(len(f),-1)
    for j,g in enumerate(groups):grp[(f.BUILDINGID.to_numpy()==g[0])&(f.FLOOR.to_numpy()==g[1])]=j
    return dict(X=T(np.concatenate([d,r,r*r],1)),inp=T(np.concatenate([d,r],1)),gt=T(gt),group=grp,phone=f.PHONEID.to_numpy(),user=f.USERID.to_numpy())
fit_a,cal_a,teach_a,val_a,test_a=map(arrays,[fit,cal,teacher,val,test])

## Generative fingerprint likelihood

For each surveyed location, model detection as Bernoulli and detected signal strength as Gaussian, using hierarchical smoothing. With a uniform survey-location prior, $p_T(z\mid x)\propto\exp(\log p(x\mid z)/T)$. Calibration selects $T$ before test. The expanded likelihood is algebraically checked against its direct expression.

In [ ]:
# Independent Bernoulli detection and conditional Gaussian RSSI, smoothed toward floor and global statistics.
d,r=encode(fit)
def stats(index,size):
    n=np.bincount(index,minlength=size)[:,None];c=np.zeros((size,A));s=c.copy();ss=c.copy();np.add.at(c,index,d);np.add.at(s,index,r);np.add.at(ss,index,r*r);return n,c,s,ss
n,c,s,ss=stats(labels,S);ng,cg,sg,ssg=stats(gi[labels],G);C=c.sum(0);gp=(C+.5)/(len(d)+1);gm=s.sum(0)/(C+5);gm2=(ss.sum(0)+5)/(C+5)
pg=(cg+20*gp)/(ng+20);mg=(sg+10*gm)/(cg+10);mg2=(ssg+10*gm2)/(cg+10)
p=np.clip((c+8*pg[gi])/(n+8),1e-5,1-1e-5);mu=(s+8*mg[gi])/(c+8);v=np.clip((ss+8*mg2[gi])/(c+8)-mu*mu,.15**2,9.)
base=np.log1p(-p).sum(1);coef=np.concatenate([np.log(p)-np.log1p(-p)-.5*np.log(2*np.pi*v)-.5*mu*mu/v,mu/v,-.5/v],1);BASE=T(base);COEF=T(coef)
def raw(a):return a['X']@COEF.T+BASE
with torch.no_grad():
    logits=raw(cal_a);temps=[1.,2.,4.,8.,16.,32.,64.];scores=[float(((torch.softmax(logits/h,1)@XY-cal_a['gt'])**2).sum(1).mean()) for h in temps];TEMP=temps[int(np.argmin(scores))]
d0=cal_a['X'][0,:A].cpu().numpy();r0=cal_a['X'][0,A:2*A].cpu().numpy();check=(d0*np.log(p[0])+(1-d0)*np.log1p(-p[0])+d0*(-.5*np.log(2*np.pi*v[0])-.5*(r0-mu[0])**2/v[0])).sum();assert np.isclose(float(logits[0,0]),check,atol=.02,rtol=1e-5)
print('Calibrated likelihood temperature:',TEMP)

## Neural proposal (GPU)

Train a classifier over surveyed locations from RSSI and detection masks. Set $q=(1-\epsilon)q_{\rm neural}+\epsilon/S$ for full support. Proposal temperature and defensive mixture are calibrated using exact model probabilities on development data only.

In [ ]:
model=nn.Sequential(nn.Linear(2*A,256),nn.LayerNorm(256),nn.ReLU(),nn.Dropout(.1),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,S)).to(DEV);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001);y=T(labels,torch.long)
for epoch in range(EPOCHS):
    model.train();loss_sum=0
    for ix in torch.randperm(len(y),device=DEV).split(256):
        loss=nn.functional.cross_entropy(model(fit_a['inp'][ix]),y[ix]);opt.zero_grad();loss.backward();opt.step();loss_sum+=float(loss.detach())*len(ix)
    if epoch==0 or (epoch+1)%5==0:print('Epoch',epoch+1,'CE',loss_sum/len(y))
model.eval()
with torch.no_grad():
    pc=torch.softmax(logits/TEMP,1);qc=model(cal_a['inp']);candidates=[(h,e) for h in [.75,1.,1.5,2.,3.,4.] for e in [.05,.15,.30]];sc=[float(-(pc*((1-e)*torch.softmax(qc/h,1)+e/S).log()).sum(1).mean()) for h,e in candidates];QT,EPS=candidates[int(np.argmin(sc))]
@torch.no_grad()
def proposal(a):return (1-EPS)*torch.softmax(model(a['inp'])/QT,1)+EPS/S
@torch.no_grad()
def prepare(a):
    lp=torch.log_softmax(raw(a)/TEMP,1);p=lp.exp();return dict(**a,lp=lp,p=p,q=proposal(a),ref=p@XY)
# Fit and calibration end here. Test posterior is used for offline evaluation only, never for deployable policy inputs.
teach_a,val_a,test_a=map(prepare,[teach_a,val_a,test_a]);assert torch.all(test_a['q']>0) and torch.allclose(test_a['p'].sum(1),torch.ones(len(test),device=DEV),atol=1e-5)
print('Proposal temperature and defensive mass:',QT,EPS)

## Features and exact bounded integer optimization

The features use only the observation and proposal, never target posterior quantities. For risk curves with decreasing marginal gains, exact greedy allocation suffices; otherwise dynamic programming solves the integer problem. A tiny exhaustive enumeration checks the DP.

In [ ]:
@torch.no_grad()
def features(a,q=None):
    q=proposal(a) if q is None else q;m=q@XY;var=(q@(XY*XY)-m*m).clamp_min(0).sum(1);ent=-(q*q.log()).sum(1);top=q.topk(2,1).values;d=a['inp'][:,:A];r=a['inp'][:,A:];count=d.sum(1);mean=r.sum(1)/count.clamp_min(1);sd=((r*r).sum(1)/count.clamp_min(1)-mean*mean).clamp_min(0).sqrt();bf=(q@GROUP).max(1).values
    return torch.stack([count,mean,sd,var.sqrt(),ent,top[:,0],top[:,0]-top[:,1],bf,(q*q).sum(1),m[:,0],m[:,1]],1).cpu().numpy()
@njit
def dp(risk,budget,kmin):
    n,L=risk.shape;extra=budget-n*kmin;mx=L-1;prev=np.full(extra+1,np.inf);prev[0]=0.;back=np.full((n,extra+1),-1,np.int32)
    for i in range(n):
        cur=np.full(extra+1,np.inf)
        for b in range(max(0,extra-(n-i-1)*mx),min(extra,(i+1)*mx)+1):
            for k in range(max(0,b-i*mx),min(mx,b)+1):
                z=prev[b-k]+risk[i,k]
                if z<cur[b]:cur[b]=z;back[i,b]=k
        prev=cur
    out=np.empty(n,np.int32);b=extra
    for i in range(n-1,-1,-1):k=back[i,b];out[i]=k+kmin;b-=k
    return out
import itertools
sm=np.array([[7.,4.,5.],[2.,6.,1.],[3.,1.,8.]]);k=dp(sm,5,1);assert sum(k)==5 and np.isclose(sum(sm[i,j-1] for i,j in enumerate(k)),min(sum(sm[i,j-1] for i,j in enumerate(z)) for z in itertools.product(range(1,4),repeat=3) if sum(z)==5))
def batches(n,seed):
    a=np.random.default_rng(seed).permutation(n);return [a[i:i+BATCH] for i in range(0,n,BATCH)]
@njit
def greedy(r,budget,kmin):
    n,L=r.shape;k=np.zeros(n,np.int32)
    for _ in range(budget-n*kmin):
        best=-1;gain=-np.inf
        for i in range(n):
            j=k[i]
            if j<L-1:
                z=r[i,j]-r[i,j+1]
                if z>gain:best=i;gain=z
        k[best]+=1
    return k+kmin
# Greedy is exact for diminishing marginal gains. Otherwise use unrestricted DP.
# This avoids charging the cheap baseline for an unnecessarily slow optimizer.
def alloc(r,bs):
    out=np.zeros(len(r),np.int32)
    for ix in bs:
        rr=np.ascontiguousarray(r[ix],dtype=np.float64);g=rr[:,:-1]-rr[:,1:]
        solver=greedy if np.all(g[:,:-1]>=g[:,1:]) else dp
        out[ix]=solver(rr,len(ix)*KAVG,KMIN)
    assert np.all((out>=KMIN)&(out<=KMAX)) and all(out[ix].sum()==len(ix)*KAVG for ix in bs);return out
def proxies(f):
    z={'uniform':np.ones(len(f)),'proposal_spread':f[:,3]+.1,'proposal_entropy':f[:,4]+.1,'inverse_AP_count':1/np.sqrt(f[:,0]+1),'proposal_ambiguity':1-f[:,5]+.05,'floor_uncertainty':1-f[:,7]+.05};return {k:s[:,None]**2/KS[None,:] for k,s in z.items()}

## Independent Monte Carlo risk banks

Each repetition draws independent proposal samples. For every prefix $K$, compute $\hat\mu_K=\sum_{k\le K} w_k z_k/\sum_{k\le K}w_k$ using stable log-cumulative sums. Exact probabilities are used to accelerate the offline risk-bank computation only. All policies are compared on the same fresh draws.

In [ ]:
# Prefix SNIS using log-cumulative sums; exact model posterior only for offline teacher/reference computation.
shift=XY.min(0).values-1;logxy=(XY-shift).log()
@torch.no_grad()
def bank(a,reps,seed,policies=None):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(a['q']);R=np.empty((n,len(KS)),np.float64);names=list(policies or {});err={k:np.empty((reps,n),np.float32) for k in names};physical={k:np.empty((reps,n),np.float32) for k in names}
    for st in range(0,n,8):
        en=min(n,st+8);q=a['q'][st:en];m=len(q);cdf=q.cumsum(1);cdf[:,-1]=1;u=torch.rand((m,reps*KMAX),generator=gen,device=DEV);idx=torch.searchsorted(cdf.contiguous(),u.contiguous()).reshape(m,reps,KMAX).clamp_max(S-1);lw=(a['lp'][st:en]-q.log()).gather(1,idx.reshape(m,-1)).reshape(m,reps,KMAX);ld=torch.logcumsumexp(lw,2);est=torch.stack([(torch.logcumsumexp(lw+logxy[idx,j],2)-ld).exp()+shift[j] for j in range(2)],-1);e=((est-a['ref'][st:en,None,None,:])**2).sum(-1)/2;R[st:en]=e[:,:,KMIN-1:].mean(1).cpu().numpy()
        if names:
            pe=((est-a['gt'][st:en,None,None,:])**2).sum(-1).sqrt();rr=torch.arange(m,device=DEV)
            for k in names:
                kk=T(policies[k][st:en]-1,torch.long);err[k][:,st:en]=e[rr,:,kk].T.cpu().numpy();physical[k][:,st:en]=pe[rr,:,kk].T.cpu().numpy()
        if st==0 or en==n or st%400==0:print('Bank',seed, en,'/',n,flush=True)
    assert np.isfinite(R).all() and all(np.isfinite(z).all() for z in err.values());return R,err,physical
ix=T([0,1,2],torch.long);lw=T([-2.,-4.,-1.]);est=(torch.logsumexp(lw[:,None]+logxy[ix],0)-torch.logsumexp(lw,0)).exp()+shift;assert torch.allclose(est,torch.softmax(lw,0)@XY[ix],atol=.001,rtol=.0001)

## Risk learning and frozen baseline selection

Fit an approximate positive finite-sample curve $\widehat R_i(K)=\exp(a_i)(K/48)^{-\beta_i}$ from teacher repetitions. Two gradient-boosted regressors predict $a_i,\beta_i$. A separate validation bank selects the strongest cheap comparator and shrinkage coefficient. No test-driven changes.

In [ ]:
ft,fv,fx=[features(a,a['q']) for a in [teach_a,val_a,test_a]];sync();teacher_start=time.perf_counter();Rt,_,_=bank(teach_a,NTEACH,SEED+100)
x=np.log(KS/KAVG);D=np.stack([np.ones_like(x),-x],1);Y=np.log(np.maximum(Rt,1e-8));pars=Y@np.linalg.pinv(D).T;pars[:,1]=np.clip(pars[:,1],.05,2.5);pars[:,0]=np.mean(Y+pars[:,1,None]*x,1)
reg=[HistGradientBoostingRegressor(max_iter=140,max_leaf_nodes=15,min_samples_leaf=15,l2_regularization=2,random_state=SEED+i).fit(ft,pars[:,i]) for i in range(2)]
sync(); teacher_seconds=time.perf_counter()-teacher_start
np.savez_compressed(OUT/'teacher_risk_curves.npz',K=KS,features=ft,risk=Rt)
def predict(f):return np.exp(reg[0].predict(f)[:,None]-np.clip(reg[1].predict(f),.05,2.5)[:,None]*x)
def blend(a,b,alpha,bs):
    z=(1-alpha)*a+alpha*b;out=np.floor(z).astype(np.int32)
    for ix in bs:
        left=int(len(ix)*KAVG-out[ix].sum());order=ix[np.argsort(-(z[ix]-out[ix]),kind='stable')];out[order[:left]]+=1
    assert all(out[ix].sum()==len(ix)*KAVG for ix in bs) and np.all((out>=KMIN)&(out<=KMAX));return out
bv=batches(len(fv),SEED+200);va={k:alloc(r,bv) for k,r in proxies(fv).items()};va['learned']=alloc(predict(fv),bv);Rv,_,_=bank(val_a,NVAL,SEED+101)
def value(r,k):return float(r[np.arange(len(k)),k-KMIN].mean())
BASELINE=min((k for k in va if k!='learned'),key=lambda k:value(Rv,va[k]));alphas=[0,.25,.5,.75,1.];avs=[value(Rv,blend(va[BASELINE],va['learned'],a,bv)) for a in alphas];ALPHA=alphas[int(np.argmin(avs))]
pd.DataFrame([{'policy':k,'risk':value(Rv,v)} for k,v in va.items()]+[{'policy':f'alpha_{a}','risk':v} for a,v in zip(alphas,avs)]).to_csv(OUT/'validation_selection.csv',index=False);print('Frozen comparator and alpha:',BASELINE,ALPHA)

## Full official test and oracle banks

The two oracle banks are independent; the pooled oracle is optimized on their mean curves. The third independent evaluation bank is used for all matched-budget results. Oracle methods are privileged offline diagnostics, not online methods.

In [ ]:
bx=batches(len(fx),SEED+201);pol={k:alloc(r,bx) for k,r in proxies(fx).items()};pol['learned']=alloc(predict(fx),bx);pol['shrunk']=blend(pol[BASELINE],pol['learned'],ALPHA,bx)
with torch.no_grad():
    p,q,ref=test_a['p'],test_a['q'],test_a['ref'];tau2=((p*p/q)*((XY[None]-ref[:,None])**2).sum(-1)/2).sum(1).cpu().numpy()
pol['asymptotic_oracle']=alloc(tau2[:,None]/KS,bx)
RA,_,_=bank(test_a,NOPT,SEED+102);np.save(OUT/'oracle_bank_A.npy',RA)
RB,_,_=bank(test_a,NOPT,SEED+103);np.save(OUT/'oracle_bank_B.npy',RB)
pol['oracle_A']=alloc(RA,bx);pol['oracle_B']=alloc(RB,bx);pol['finite_oracle']=alloc((RA+RB)/2,bx)
RE,errs,locerr=bank(test_a,NEVAL,SEED+104,pol)

## Statistical results

Report paired Monte Carlo confidence intervals conditional on the fixed test set. Test records may share users, phones or spatial conditions, so these intervals must not be described as independent-deployment confidence intervals. Device/user strata are descriptive. Compare the direct alternatives independently of the matched target-evaluation budget.

In [ ]:
# Intervals describe paired Monte Carlo uncertainty conditional on this fixed real test set; not a population CI.
b=errs[BASELINE].mean(1).astype(float);bm=b.mean();um=float(errs['uniform'].mean());bb=np.array([errs[BASELINE][:,ix].mean() for ix in bx]);rows=[];br=[]
for k,e in errs.items():
    a=e.mean(1).astype(float);am=a.mean();gain=1-am/bm;infl=-(a-am)/bm+am*(b-bm)/bm**2;h=t.ppf(.975,len(a)-1)*infl.std(ddof=1)/np.sqrt(len(a));mb=np.array([e[:,ix].mean() for ix in bx]);harm=(mb-bb)/np.maximum(bb,1e-12)
    rows.append(dict(method=k,target_evals_per_record=KAVG,posterior_mse=am,gain_vs_uniform_pct=100*(1-am/um),gain_vs_primary_pct=100*gain,ci_low_pct=100*(gain-h),ci_high_pct=100*(gain+h),mean_coordinate_error=float(locerr[k].mean()),median_coordinate_error=float(np.median(locerr[k])),harmful_batch_fraction=float((harm>0).mean()),worst_decile_loss=float(np.sort(harm)[-max(1,math.ceil(.1*len(harm))):].mean())))
    for i,ix in enumerate(bx):br.append(dict(method=k,batch=i,n=len(ix),risk=mb[i],gain_pct=-100*harm[i]))
summary=pd.DataFrame(rows).sort_values('posterior_mse');summary.to_csv(OUT/'method_summary.csv',index=False);pd.DataFrame(br).to_csv(OUT/'batch_results.csv',index=False);display(summary.round(4))
# Direct exact reference, neural proposal mean, and conventional 5NN; not forced into the sampling budget comparison.
@torch.no_grad()
def knn(a):
    ix=torch.cdist(a['inp'],fit_a['inp']).topk(5,largest=False).indices;return fit_a['gt'][ix].mean(1),GROUP[T(labels,torch.long)[ix]].sum(1).argmax(1)
with torch.no_grad():
    direct=[];kn=[knn({'inp':a}) for a in test_a['inp'].split(64)];pairs=[('exact_discrete_posterior',test_a['p']@XY,(test_a['p']@GROUP).argmax(1),S),('direct_neural_proposal',test_a['q']@XY,(test_a['q']@GROUP).argmax(1),0),('direct_5NN',torch.cat([a[0] for a in kn]),torch.cat([a[1] for a in kn]),0)]
    for k,est,grp,cost in pairs:
        e=(est-test_a['gt']).norm(dim=1);good=test_a['group']>=0;direct.append(dict(method=k,target_evals_per_record=cost,posterior_mse=float(((est-test_a['ref'])**2).sum(1).mean()/2),mean_coordinate_error=float(e.mean()),median_coordinate_error=float(torch.quantile(e,.5)),building_floor_accuracy=float((grp.cpu().numpy()[good]==test_a['group'][good]).mean()) if good.any() else None,group_coverage=float(good.mean())))
direct=pd.DataFrame(direct);direct.to_csv(OUT/'direct_comparators.csv',index=False);display(direct.round(4))
ss=summary.set_index('method');o=ss.loc['finite_oracle'];l=ss.loc['learned'];gate=dict(primary_baseline=BASELINE,oracle_gain_pct=float(o.gain_vs_primary_pct),oracle_ci_low_pct=float(o.ci_low_pct),oracle_ci_high_pct=float(o.ci_high_pct),oracle_gate_pass=bool(o.gain_vs_primary_pct>=2 and o.ci_low_pct>0),oracle_A_B_mean_abs_allocation_diff=float(np.abs(pol['oracle_A']-pol['oracle_B']).mean()),learned_gain_pct=float(l.gain_vs_primary_pct),learned_ci_low_pct=float(l.ci_low_pct),shrink_alpha=ALPHA,captured_oracle_gain_fraction=float(l.gain_vs_primary_pct/o.gain_vs_primary_pct) if o.gain_vs_primary_pct>0 else None,interpretation='Conditional Monte Carlo uncertainty only; approximate estimated oracle, not guaranteed upper bound.');print(json.dumps(gate,indent=2));(OUT/'gate_summary.json').write_text(json.dumps(gate,indent=2))
strata=[]
for field in ['phone','user']:
    for g in np.unique(test_a[field]):
        ix=np.flatnonzero(test_a[field]==g);den=float(errs[BASELINE][:,ix].mean())
        for k in ['finite_oracle','learned','shrunk']:strata.append(dict(field=field,group=str(g),n=len(ix),method=k,gain_pct=100*(1-float(errs[k][:,ix].mean())/max(den,1e-12))))
pd.DataFrame(strata).to_csv(OUT/'group_sensitivity.csv',index=False)

## Actual online runtime (no cached full posterior)

Time actual computation of only the assigned target likelihood terms for sampling methods. Exact summation, proposal-only prediction and 5NN are independently timed. Use synchronized warm runs on the same benchmark batch. These timings exclude input encoding and do not imply a production latency guarantee.

In [ ]:
@torch.no_grad()
def online_sample(a,q,k,seed):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(k);mx=int(max(k));idx=torch.multinomial(q,mx,replacement=True,generator=gen);mask=torch.arange(mx,device=DEV)[None,:]<T(k,torch.long)[:,None];r,c=torch.where(mask);j=idx[r,c];lw=((a['X'][r]*COEF[j]).sum(1)+BASE[j])/TEMP-q[r,j].log();w=torch.full((n,mx),-torch.inf,device=DEV);w[r,c]=lw;return (torch.softmax(w,1)[:,:,None]*XY[idx]).sum(1)
ii=torch.arange(min(8,len(test)),device=DEV);jj=ii%S;assert torch.allclose((test_a['X'][ii]*COEF[jj]).sum(1)+BASE[jj],raw(test_a)[ii,jj],atol=.02,rtol=1e-5)
ri=bx[0];a={k:test_a[k][ri] for k in ['X','inp']};b0=[np.arange(len(ri))]
@torch.no_grad()
def online(k,seed):
    if k=='exact_discrete_posterior':return torch.softmax(raw(a)/TEMP,1)@XY
    if k=='direct_5NN':return knn(a)[0]
    q=proposal(a)
    if k=='direct_neural_proposal':return q@XY
    if k=='uniform':counts=np.full(len(ri),KAVG,np.int32)
    else:
        f=features(a,q)
        if k=='learned':counts=alloc(predict(f),b0)
        elif k=='shrunk':counts=blend(alloc(proxies(f)[BASELINE],b0),alloc(predict(f),b0),ALPHA,b0)
        else:counts=alloc(proxies(f)[k],b0)
    return online_sample(a,q,counts,seed)
rt=[]
for k in dict.fromkeys(['uniform',BASELINE,'learned','shrunk','exact_discrete_posterior','direct_neural_proposal','direct_5NN']):
    online(k,SEED);sync()
    for j in range(12 if not TEST_FIXTURE else 2):
        sync();st=time.perf_counter();e=online(k,SEED+j+500);sync();ms=1000*(time.perf_counter()-st);assert torch.isfinite(e).all();rt.append(dict(method=k,repeat=j,batch_size=len(ri),milliseconds=ms))
rt=pd.DataFrame(rt);rt.to_csv(OUT/'runtime_trials.csv',index=False);runt=rt.groupby('method').milliseconds.agg(['median','min','max']).reset_index();runt.to_csv(OUT/'runtime_summary.csv',index=False);display(runt.round(3))

## Example deployment-style batch

Run the frozen validation-selected policy on RSSI alone, then attach recorded coordinates for evaluation. This is a working batch-inference demonstration on held-out real observations.

In [ ]:
# Estimate locations from RSSI alone, using the fixed validation-selected shrinkage policy.
# Ground-truth coordinates are joined only AFTER prediction for demonstration/evaluation.
with torch.no_grad():
    q=proposal(a);ff=features(a,q);kk=blend(alloc(proxies(ff)[BASELINE],b0),alloc(predict(ff),b0),ALPHA,b0)
    point=online_sample(a,q,kk,SEED+600).cpu().numpy()+origin
    truth=test_a['gt'][ri].cpu().numpy()+origin
    demo=pd.DataFrame({'official_test_record':ri,'samples':kk,'estimated_x':point[:,0],'estimated_y':point[:,1],'recorded_x':truth[:,0],'recorded_y':truth[:,1],'coordinate_error':np.linalg.norm(point-truth,axis=1)})
    demo.to_csv(OUT/'example_online_batch.csv',index=False);display(demo.head(10).round(3))
print('Risk teacher + regressor elapsed seconds (excludes common model/proposal fit):',teacher_seconds)

## Figures and reproducible export

Save all metrics and plots before downloading the archive. The original data is not redistributed in the archive.

In [ ]:
pd.DataFrame({'record':np.arange(len(fx)),**pol}).to_csv(OUT/'allocations.csv',index=False);np.savez_compressed(OUT/'risk_curves.npz',K=KS,oracle_A=RA,oracle_B=RB,independent_evaluation=RE,predicted=predict(fx));np.savez_compressed(OUT/'test_mc_metrics.npz',**{f'risk_{k}':v for k,v in errs.items()},**{f'coordinate_{k}':v for k,v in locerr.items()})
def save(name):plt.tight_layout();plt.savefig(OUT/name,dpi=180);plt.show();plt.close()
plt.figure(figsize=(9,5));z=summary.sort_values('gain_vs_primary_pct');g=z.gain_vs_primary_pct.to_numpy();plt.errorbar(g,np.arange(len(z)),xerr=np.maximum(0,np.stack([g-z.ci_low_pct,z.ci_high_pct-g])),fmt='o',capsize=3);plt.yticks(np.arange(len(z)),z.method);plt.axvline(0,color='gray');plt.axvline(2,color='gray',linestyle=':');plt.xlabel('MSE reduction vs frozen cheap comparator (%)');save('01_gain.png')
plt.figure(figsize=(8,5));plt.plot(KS,RE.mean(0),label='Independent empirical');plt.plot(KS,(tau2[:,None]/KS).mean(0),label='Asymptotic');plt.plot(KS,predict(fx).mean(0),label='Learned');plt.yscale('log');plt.xlabel('Samples');plt.ylabel('Posterior MSE (projected units squared)');plt.legend();save('02_risk_curves.png')
plt.figure(figsize=(6,5));plt.scatter(pol['oracle_A'],pol['oracle_B'],s=9,alpha=.4);plt.plot([KMIN,KMAX],[KMIN,KMAX],'--');plt.xlabel('Oracle A allocation');plt.ylabel('Oracle B allocation');save('03_oracle_stability.png')
plt.figure(figsize=(8,5));names=list(dict.fromkeys(['uniform',BASELINE,'learned','shrunk','finite_oracle']));plt.barh(names+direct.method.tolist(),[float(locerr[k].mean()) for k in names]+direct.mean_coordinate_error.tolist());plt.xlabel('Mean horizontal error (native projected units)');save('04_localization.png')
plt.figure(figsize=(8,4));plt.barh(runt.method,runt['median']);plt.xlabel('Median online latency (ms/batch)');save('05_latency.png')
plt.figure(figsize=(7,5));g=test_a['gt'].cpu().numpy();r=test_a['ref'].cpu().numpy();plt.scatter(g[:,0],g[:,1],s=8,alpha=.5,label='Recorded positions');plt.scatter(r[:,0],r[:,1],s=8,alpha=.5,label='Exact posterior mean');plt.axis('equal');plt.legend();plt.xlabel('Centered projected x');plt.ylabel('Centered projected y');save('06_map.png')
import joblib
joblib.dump(reg,OUT/'risk_model.joblib');torch.save(model.cpu().state_dict(),OUT/'proposal.pt');np.savez_compressed(OUT/'target_model.npz',nodes=nodes,origin=origin,coef=coef,base=base,temperature=TEMP,proposal_temperature=QT,defensive_mass=EPS);np.savez_compressed(OUT/'development_split_indices.npz',fit=parts[0],cal=parts[1],teacher=parts[2],validation=parts[3])
manifest=dict(real_data=not TEST_FIXTURE,dataset='UJIIndoorLoc',sha256=sha,split=split_type,train_size=len(train),test_size=len(test),fit_size=len(fit),cal_size=len(cal),teacher_size=len(teacher),val_size=len(val),nodes=S,seed=SEED,device=str(DEV),torch_version=torch.__version__,oracle_reps_each=NOPT,evaluation_reps=NEVAL,teacher_reps=NTEACH,validation_reps=NVAL,kmin=KMIN,kmax=KMAX,kavg=KAVG,batch_size=BATCH,temperature=TEMP,proposal_temperature=QT,defensive_mass=EPS,baseline=BASELINE,alpha=ALPHA,coordinate_units='native projected units, not verified physical metres',ci='conditional Monte Carlo only',risk_teacher_fit_seconds=teacher_seconds,elapsed_seconds_before_archive=time.perf_counter()-start_total,proposal_architecture='2A-256-LayerNorm-ReLU-Dropout-128-ReLU-S',source_urls=URLS);(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=OUT.parent/('WIFI_FIXTURE_ONLY.zip' if TEST_FIXTURE else 'UJIIndoorLoc_Finite_Sample_Risk_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('Results archive:',archive)
if not TEST_FIXTURE:
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:pass

## Interpret the results without changing the protocol

- If the primary oracle gate fails, report that useful oracle headroom was **not demonstrated** in this pipeline; inspect oracle bank stability and the risk curves.
- If the oracle passes but learned risk does not beat the selected cheap policy, there is a gap between privileged allocation and predictable allocation. Do not claim a learned advantage.
- If the learned or shrunk policy improves posterior estimation, check localization error and end-to-end latency separately. An improvement in the posterior statistic need not translate into better positioning.
- If exact enumeration or direct regression is faster and equally or more accurate, the practical motivation for adaptive importance sampling in this pipeline is limited. Report this result rather than omitting those baselines.
- The held-out benchmark is a retrospective real-data study, not a prospective field deployment. A successful result motivates testing more expensive continuous or higher-dimensional localization models, not a claim of universal benefit.

Dataset citation: J. Torres-Sospedra et al., UJIIndoorLoc, UCI Machine Learning Repository (2014), https://archive.ics.uci.edu/dataset/310/ujiindoorloc.